# BO7 Custom YOLOv8 Training (GPU)
**Trainiert ein Custom-Modell fuer Black Ops 7 / Warzone Gegnererkennung**

## Anleitung:
1. Laufzeittyp auf **GPU** stellen (Laufzeit → Laufzeittyp aendern → T4 GPU)
2. Alle Zellen der Reihe nach ausfuehren (Strg+F9)
3. Am Ende die ONNX-Datei runterladen

---

## Schritt 1: Setup & GPU Check

In [ ]:
!pip install ultralytics -q
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA verfuegbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("FEHLER: Keine GPU! Geh zu Laufzeit -> Laufzeittyp aendern -> GPU")

## Schritt 2: Dateien hochladen
Lade diese 2 Dateien hoch (aus deinem `xbox-vision-aimbot` Ordner):
1. `labels_only.zip` (1.2 MB - Labels)
2. Dein **Screenshots-Ordner** als ZIP (die resized 320x320 Bilder)

### Option A: Direkt hochladen (einfacher, aber langsamer bei grossen Dateien)

In [ ]:
from google.colab import files
import os

print("=== Lade labels_only.zip hoch ===")
uploaded = files.upload()
print("Labels hochgeladen!")

In [ ]:
print("=== Lade deine Screenshots als ZIP hoch ===")
print("(Der Ordner mit den 5120 resized .jpg Bildern)")
uploaded2 = files.upload()
print("Bilder hochgeladen!")

### Option B: Von Google Drive (schneller bei grossen Dateien)
Falls du die Dateien lieber ueber Google Drive hochladen willst, fuehre stattdessen diese Zelle aus:

In [ ]:
# NUR ausfuehren wenn du Option B (Google Drive) nutzen willst!
# Kommentiere die naechsten 3 Zeilen ein:

#from google.colab import drive
#drive.mount('/content/drive')
#print("Google Drive verbunden! Kopiere deine Dateien nach /content/")

# Dann passe die Pfade an:
#!cp /content/drive/MyDrive/labels_only.zip /content/
#!cp /content/drive/MyDrive/DEIN_BILDER_ORDNER.zip /content/

## Schritt 3: Dataset vorbereiten

In [ ]:
import zipfile, os, glob, shutil

# Labels entpacken
print("Entpacke Labels...")
with zipfile.ZipFile("labels_only.zip", "r") as z:
    z.extractall("/content/")

# Dataset-Ordner erstellen
os.makedirs("/content/dataset_bo7_v2/train/images", exist_ok=True)
os.makedirs("/content/dataset_bo7_v2/val/images", exist_ok=True)

# Bilder-ZIP finden und entpacken
zip_files = [f for f in os.listdir("/content/") if f.endswith(".zip") and f != "labels_only.zip"]
if zip_files:
    img_zip = zip_files[0]
    print(f"Entpacke Bilder aus {img_zip}...")
    with zipfile.ZipFile(img_zip, "r") as z:
        z.extractall("/content/temp_images/")
    
    # Alle .jpg Dateien finden und ins Dataset kopieren
    all_jpgs = glob.glob("/content/temp_images/**/*.jpg", recursive=True)
    print(f"Gefunden: {len(all_jpgs)} Bilder")
    
    # Bilder den Labels zuordnen (train vs val)
    train_labels = set(os.path.splitext(f)[0] for f in os.listdir("/content/dataset_bo7_v2/train/labels/"))
    val_labels = set(os.path.splitext(f)[0] for f in os.listdir("/content/dataset_bo7_v2/val/labels/"))
    
    train_count = 0
    val_count = 0
    for jpg_path in all_jpgs:
        fname = os.path.splitext(os.path.basename(jpg_path))[0]
        if fname in train_labels:
            shutil.copy2(jpg_path, f"/content/dataset_bo7_v2/train/images/{fname}.jpg")
            train_count += 1
        elif fname in val_labels:
            shutil.copy2(jpg_path, f"/content/dataset_bo7_v2/val/images/{fname}.jpg")
            val_count += 1
    
    print(f"Train: {train_count} Bilder | Val: {val_count} Bilder")
else:
    print("FEHLER: Keine Bilder-ZIP gefunden! Lade deine Screenshots als ZIP hoch.")

# data.yaml anpassen
with open("/content/dataset_bo7_v2/data.yaml", "w") as f:
    f.write("path: /content/dataset_bo7_v2\n")
    f.write("train: train/images\n")
    f.write("val: val/images\n")
    f.write("\n")
    f.write("nc: 2\n")
    f.write("names:\n")
    f.write("  0: player\n")
    f.write("  1: head\n")

print("\nDataset bereit!")

## Schritt 4: Training starten
**YOLOv8s** (Small) statt Nano — deutlich genauer, trotzdem schnell genug fuer Echtzeit.

Erwartete Trainingszeit: **20-40 Minuten** auf T4 GPU

In [ ]:
from ultralytics import YOLO
import time

print("=== BO7 CUSTOM MODEL - GPU TRAINING ===")
start = time.time()

# YOLOv8s (Small) - bessere Genauigkeit als Nano
model = YOLO("yolov8s.pt")

results = model.train(
    data="/content/dataset_bo7_v2/data.yaml",
    epochs=100,
    imgsz=640,           # Groessere Bilder beim Training = bessere Erkennung
    batch=32,            # GPU kann grosse Batches
    device=0,            # GPU
    workers=4,
    patience=15,         # Early Stopping nach 15 Epochen ohne Verbesserung
    save=True,
    project="/content/runs",
    name="bo7_gpu_v4",
    exist_ok=True,
    
    # Optimierte Hyperparameter
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    
    # Starke Augmentation fuer bessere Generalisierung
    mosaic=1.0,
    mixup=0.15,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,         # Leichte Rotation
    translate=0.15,
    
    # Verlust-Gewichtung: Mehr Fokus auf Klassifikation
    box=7.5,
    cls=1.5,             # Hoeher = bessere Unterscheidung player vs head
    dfl=1.5,
    
    verbose=True,
    plots=True,
)

elapsed = time.time() - start
print(f"\nTraining fertig in {elapsed/60:.1f} Minuten!")

## Schritt 5: Ergebnisse anschauen

In [ ]:
from IPython.display import Image, display
import os

run_dir = "/content/runs/bo7_gpu_v4"

# Ergebnisse anzeigen
for img_name in ["results.png", "confusion_matrix.png", "val_batch0_pred.png"]:
    img_path = os.path.join(run_dir, img_name)
    if os.path.exists(img_path):
        print(f"\n=== {img_name} ===")
        display(Image(filename=img_path, width=800))

## Schritt 6: ONNX Export
Exportiert das beste Modell als ONNX (320px) fuer deinen Aimbot.

In [ ]:
from ultralytics import YOLO
import shutil, os

best_pt = "/content/runs/bo7_gpu_v4/weights/best.pt"

# === Export 1: 320px (Schnell - fuer Aimbot) ===
print("=== Export: 320px ONNX (fuer Aimbot) ===")
model = YOLO(best_pt)
model.export(format="onnx", imgsz=320, simplify=True, half=False)

onnx_src = best_pt.replace(".pt", ".onnx")
onnx_320 = "/content/bo7_custom_v4_320.onnx"
shutil.copy2(onnx_src, onnx_320)
sz = os.path.getsize(onnx_320) / (1024*1024)
print(f"Gespeichert: bo7_custom_v4_320.onnx ({sz:.1f} MB)")

# === Export 2: 640px (Genauer - optional) ===
print("\n=== Export: 640px ONNX (genauer, langsamer) ===")
model2 = YOLO(best_pt)
model2.export(format="onnx", imgsz=640, simplify=True, half=False)

onnx_src2 = best_pt.replace(".pt", ".onnx")
onnx_640 = "/content/bo7_custom_v4_640.onnx"
shutil.copy2(onnx_src2, onnx_640)
sz2 = os.path.getsize(onnx_640) / (1024*1024)
print(f"Gespeichert: bo7_custom_v4_640.onnx ({sz2:.1f} MB)")

print("\n=== FERTIG! Lade die ONNX-Dateien runter ===")

## Schritt 7: Download
Lade die ONNX-Dateien herunter und kopiere sie in deinen `backend/` Ordner.

In [ ]:
from google.colab import files

print("Downloading 320px model (fuer Aimbot)...")
files.download("/content/bo7_custom_v4_320.onnx")

print("Downloading 640px model (optional, genauer)...")
files.download("/content/bo7_custom_v4_640.onnx")

## Fertig!

### Naechste Schritte:
1. Kopiere `bo7_custom_v4_320.onnx` nach `C:\Users\hhgra\Desktop\xbox-vision-aimbot\backend\bo7_custom_320.onnx`
2. (Optional) Kopiere `bo7_custom_v4_640.onnx` nach `backend\bo7_custom_640.onnx`
3. Starte den Aimbot: `START_AIMBOT.bat`
4. Das Modell wird automatisch als BO7-Custom erkannt (2 Klassen: player + head)